# 12. IsolationForest 후보 목록 만들기 (v7 라벨링용)

| 항목 | 내용 |
| --- | --- |
| 입력 | `cal01_base_v6.csv` (1분), 검토 표 `domain_decisions.csv`, 정비이력 `G-02_정비이력.csv` |
| 출력 | `data/processed/if_candidates.csv` (팀이 검토할 후보 목록) |
| 이 노트북이 하는 일 | 구역마다 IsolationForest로 평소와 다른 분을 찾아 묶음으로 만들고, 이미 판단한 구간을 뺀 뒤 50개 안으로 고름 |

**왜 만드나**

- 지금 정답 1의 96%는 팀이 보지 않은 3σ 통계 후보임. v7은 모델이 고른 후보를 팀이 보고 정답을 만드는 순서로 바꿈 (#117)
- 후보를 뽑는 용도라서 LOT 전체를 한 번에 학습함. 모델 평가는 03처럼 시간순으로 함

**후보 고르는 규칙**

| 규칙 | 값 |
| --- | --- |
| 입력 변수 | 구역 센서마다 지난 값 \|Z\|, 1분 변화, 1분 표준편차 (같은 LOT 안의 지난 값만) |
| 이상으로 보는 비율 | 구역마다 상위 2% |
| 묶음 | 이어진 분을 하나로. 3분 이상 묶음을 먼저 고름 |
| 뺌 | 검토 표에 이미 판단(v2, v3, v6, v6_보류)이 있는 구간 |
| 개수 | 구역마다 최대 6개, 전체 50개 |
| 섞어 넣기 | Z-score만 울린 긴 묶음과 조용한 구간을 몇 개 더함. 팀에게는 출처를 보여 주지 않음 |

- 섞어 넣는 이유: IsolationForest가 고른 것만 라벨링하면 그 정답으로 IsolationForest를 채점할 수 없음 (놓친 이상이 생길 수 없음)
- Z9, COOLING은 통계 후보가 한 번도 없어 뺌


## 1. 공통 경로와 설정

In [ ]:
import os
import pandas as pd

# 저장소 루트 찾기: notebooks 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

# 폴더
DATA_DIR = os.path.join(ROOT, "data")                    # 원본 CSV 8개
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")             # data/raw에 두었으면 그 폴더를 씀
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")  # 만든 CSV (cal01_base_v1 ~ v6)
LABEL_DIR = os.path.join(ROOT, "data", "labels")         # 팀 검토 표 (domain_decisions.csv)
FIG_DIR = os.path.join(ROOT, "figures")                  # 그림
ENC = "utf-8-sig"

# 원본 CSV
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 읽을 가공 CSV (이름만 바꾸면 됨)
DATA_FILE = "cal01_base_v6.csv"
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)

# 만들 CSV
OUT_CSV = os.path.join(PROCESSED_DIR, "if_candidates.csv")

# 검토 표 (이미 판단한 구간을 빼는 데 씀)
DECISION_CSV = os.path.join(LABEL_DIR, "domain_decisions.csv")
USE_VERSIONS = ["v2", "v3", "v6", "v6_보류"]

# 후보 고르는 규칙
CONTAMINATION = 0.02  # 구역마다 상위 2%를 이상으로 봄
MIN_RUN = 3           # 이 길이(분) 이상 묶음을 먼저 고름
PER_ZONE = 6          # 구역마다 최대 개수
MAX_TOTAL = 50        # 전체 최대 개수
RANDOM_STATE = 42     # 프로젝트 가이드대로 고정

import numpy as np
from sklearn.ensemble import IsolationForest

## 2. 구역과 센서

In [ ]:
# 후보를 뽑을 구역 (Z9, COOLING은 통계 후보가 없어 뺌)
GROUPS = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
}

SENSORS = []
for zone in GROUPS:
    SENSORS = SENSORS + GROUPS[zone]

## 3. 데이터 읽기 (제외한 분은 뺌)

In [ ]:
columns = ["LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED"]

for sensor in SENSORS:
    columns = columns + [sensor, sensor + "__STD"]

for zone in GROUPS:
    columns = columns + [zone + "__STAT_LABEL", zone + "__DOMAIN_LABEL"]

df = pd.read_csv(BASE_CSV, encoding=ENC, usecols=columns)
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"], errors="raise")
df = df.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)

# 정비, 정지 등으로 제외한 분은 후보에서 뺌
df = df[df["EXCLUDED"] == False].reset_index(drop=True)

lots = sorted(df["LOT_NO"].unique())
print(df.shape, "/ LOT", len(lots))

## 4. 입력 변수: 지난 값 |Z|, 1분 변화, 1분 표준편차

- 같은 LOT에서 현재 분 이전까지 본 값의 평균과 표준편차로 \|Z\|를 계산함 (03의 Z-score 기준선과 같은 계산)
- 앞의 30분은 기준이 부족해서 0으로 둠
- 1분 표준편차는 1분 안에 기록이 하나뿐이면 비므로 0으로 채움

In [ ]:
for lot in lots:
    rows = df["LOT_NO"] == lot

    for sensor in SENSORS:
        values = df.loc[rows, sensor]
        previous = values.shift(1)

        past_mean = previous.expanding(min_periods=30).mean()
        past_std = previous.expanding(min_periods=30).std(ddof=0)
        past_std = past_std.replace(0, np.nan)

        df.loc[rows, sensor + "__ZPAST"] = ((values - past_mean).abs() / past_std).fillna(0)
        df.loc[rows, sensor + "__DIFF"] = (values - previous).fillna(0)

for sensor in SENSORS:
    df[sensor + "__STD"] = df[sensor + "__STD"].fillna(0)

print("입력 변수 예:", [SENSORS[0] + "__ZPAST", SENSORS[0] + "__DIFF", SENSORS[0] + "__STD"])

## 5. IsolationForest 점수

- 구역마다 그 구역 센서의 입력 변수만 넣고 LOT 전체를 한 번에 학습함 (후보 발굴용)
- `score_samples`는 낮을수록 평소와 다름. 상위 2%를 후보 분으로 표시함

In [ ]:
scores = {}
for zone in GROUPS:
    features = []
    for sensor in GROUPS[zone]:
        features = features + [sensor + "__ZPAST", sensor + "__DIFF", sensor + "__STD"]

    X = df[features]

    iso = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=RANDOM_STATE, n_jobs=-1)
    iso.fit(X)

    scores[zone + "__IF_SCORE"] = iso.score_samples(X)
    scores[zone + "__IF_FLAG"] = (iso.predict(X) == -1).astype(int)

# 구역별 점수와 표시를 한 번에 붙임
df = pd.concat([df, pd.DataFrame(scores, index=df.index)], axis=1)

for zone in GROUPS:
    print(zone, "후보 분:", int(df[zone + "__IF_FLAG"].sum()))

## 6. 이어진 분을 묶음으로

- 같은 LOT에서 1분 간격으로 이어진 후보 분을 하나의 묶음으로 봄
- 묶음마다 가장 낮은 점수, 가장 큰 지난 값 \|Z\|와 그 센서, 통계 후보 여부(STAT 1), Z-score 경보 여부(\|Z\| > 3)를 남김

In [ ]:
def make_runs(flag_column, zone):
    # 한 구역의 표시된 분을 묶음 목록으로 만듦
    runs = []

    for lot in lots:
        part = df[df["LOT_NO"] == lot]

        start = None
        previous_time = None
        members = []

        for i in part.index:
            flagged = part.loc[i, flag_column] == 1
            now = part.loc[i, "MEAS_DT"]

            # 표시가 끊기거나 1분 넘게 비면 묶음을 닫음
            continues = flagged and previous_time is not None and (now - previous_time) <= pd.Timedelta(minutes=1) and start is not None

            if flagged and not continues:
                if start is not None:
                    runs.append(members)
                start = now
                members = [i]
            elif continues:
                members.append(i)
            elif start is not None:
                runs.append(members)
                start = None
                members = []

            previous_time = now

        if start is not None:
            runs.append(members)

    rows = []
    for members in runs:
        part = df.loc[members]

        # 묶음 안에서 지난 값 |Z|가 가장 큰 센서
        best_sensor = GROUPS[zone][0]
        best_z = -1.0
        for sensor in GROUPS[zone]:
            z = part[sensor + "__ZPAST"].max()
            if z > best_z:
                best_z = z
                best_sensor = sensor

        rows.append({
            "LOT_NO": int(part["LOT_NO"].iloc[0]),
            "GRADE": part["GRADE"].iloc[0],
            "ZONE": zone,
            "SENSOR": best_sensor,
            "START": part["MEAS_DT"].min(),
            "END": part["MEAS_DT"].max() + pd.Timedelta(seconds=59),
            "MINUTES": len(part),
            "IF_SCORE": round(part[zone + "__IF_SCORE"].min(), 4),
            "MAX_ZPAST": round(best_z, 2),
            "STAT_ANY": int((part[zone + "__STAT_LABEL"] == 1).any()),
            "ZSCORE_ANY": int(best_z > 3),
        })

    return rows


if_runs = []
for zone in GROUPS:
    if_runs = if_runs + make_runs(zone + "__IF_FLAG", zone)

if_runs = pd.DataFrame(if_runs)
print("IsolationForest 묶음:", len(if_runs), "/ 3분 이상:", int((if_runs["MINUTES"] >= MIN_RUN).sum()))
if_runs.groupby("ZONE")["MINUTES"].agg(["count", "max"])

## 7. 이미 판단한 구간 빼기

검토 표(v2, v3, v6, v6_보류)에서 같은 LOT, 같은 구역 센서, 시간이 겹치는 판단이 있으면 뺌. 팀이 이미 본 구간을 다시 묻지 않기 위함

In [ ]:
decisions = pd.read_csv(DECISION_CSV, encoding=ENC)
decisions = decisions[decisions["VERSION"].isin(USE_VERSIONS)].copy()
decisions["START"] = pd.to_datetime(decisions["START"], errors="raise")
decisions["END"] = pd.to_datetime(decisions["END"], errors="raise")


def find_decided(run):
    # 이 묶음과 겹치는 판단 번호를 쉼표로 이어 돌려줌 (없으면 빈 문자열)
    same = decisions[(decisions["LOT_NO"] == run["LOT_NO"]) & decisions["SENSOR"].isin(GROUPS[run["ZONE"]])]
    overlap = same[(same["START"] <= run["END"]) & (same["END"] >= run["START"])]

    return ", ".join(overlap["DECISION_ID"])


if_runs["DECIDED"] = if_runs.apply(find_decided, axis=1)

before = len(if_runs)
if_runs = if_runs[if_runs["DECIDED"] == ""].reset_index(drop=True)
print("이미 판단한 구간을 뺌:", before, "→", len(if_runs))

## 8. 가까운 CAL01 정비 기록

후보 뒤에 처음 오는 감지 시각(`DETC_DT`)과 후보 앞에 마지막으로 끝난 정비(`END_DT`)를 시간 차이와 함께 붙임. 판단에 참고만 함

In [ ]:
maint = pd.read_csv(MAINT_CSV, encoding=ENC)
maint = maint[maint["EQP_TAG"] == "P1-CR1-CAL01"].copy()
maint["DETC_DT"] = pd.to_datetime(maint["DETC_DT"], errors="raise")
maint["END_DT"] = pd.to_datetime(maint["END_DT"], errors="raise")
maint["FAIL_MODE"] = maint["FAIL_MODE"].fillna("")


def nearest_maintenance(run):
    # 뒤에 오는 첫 감지와 앞에 끝난 마지막 정비를 짧은 글로 만듦
    after = maint[maint["DETC_DT"] >= run["END"]].sort_values("DETC_DT")
    before = maint[maint["END_DT"] <= run["START"]].sort_values("END_DT")

    after_text = ""
    if len(after) > 0:
        first = after.iloc[0]
        hours = (first["DETC_DT"] - run["END"]).total_seconds() / 3600
        after_text = first["MNT_TYPE"] + " " + str(round(hours, 1)) + "시간 뒤 " + first["FAIL_MODE"]

    before_text = ""
    if len(before) > 0:
        last = before.iloc[-1]
        hours = (run["START"] - last["END_DT"]).total_seconds() / 3600
        before_text = last["MNT_TYPE"] + " " + str(round(hours, 1)) + "시간 전"

    return pd.Series({"NEXT_MAINT": after_text.strip(), "PREV_MAINT": before_text})


if_runs[["NEXT_MAINT", "PREV_MAINT"]] = if_runs.apply(nearest_maintenance, axis=1)
if_runs.head()

## 9. 후보 고르기

1. 3분 이상 묶음을 먼저, 그 안에서는 점수가 낮은(더 다른) 순서
2. 구역마다 최대 6개
3. 전체 50개

In [ ]:
if_runs["LONG"] = (if_runs["MINUTES"] >= MIN_RUN).astype(int)
if_runs = if_runs.sort_values(["LONG", "IF_SCORE"], ascending=[False, True]).reset_index(drop=True)

picked = []
for zone in GROUPS:
    zone_runs = if_runs[if_runs["ZONE"] == zone]
    picked.append(zone_runs.head(PER_ZONE))

candidates = pd.concat(picked)
candidates = candidates.sort_values(["LONG", "IF_SCORE"], ascending=[False, True]).head(MAX_TOTAL)
candidates = candidates.sort_values(["LOT_NO", "START"]).reset_index(drop=True)
candidates["SOURCE"] = "IF"

print("고른 후보:", len(candidates))
candidates.groupby("ZONE")["MINUTES"].agg(["count", "sum"])

## 10. 섞어 넣을 후보: Z-score만 울린 긴 묶음, 조용한 구간

- Z-score만: 통계 후보(STAT 1)이면서 IsolationForest가 표시하지 않은 분의 묶음 중 가장 긴 것, 구역마다 1개
- 조용한 구간: 모든 구역이 통계 후보도 아니고 IsolationForest 표시도 없는 분에서 무작위로 고른 10분
- 팀에게 올릴 때는 `SOURCE`를 빼고 올림

In [ ]:
# 컬럼을 많이 더해 조각난 프레임을 한 번 정리함
df = df.copy()

# Z-score만 울린 묶음
for zone in GROUPS:
    df[zone + "__ZONLY"] = ((df[zone + "__STAT_LABEL"] == 1) & (df[zone + "__IF_FLAG"] == 0)).astype(int)

z_only = []
for zone in GROUPS:
    z_only = z_only + make_runs(zone + "__ZONLY", zone)

z_only = pd.DataFrame(z_only)
z_only["DECIDED"] = z_only.apply(find_decided, axis=1)
z_only = z_only[z_only["DECIDED"] == ""]
z_only = z_only.sort_values("MINUTES", ascending=False)

z_picked = []
for zone in GROUPS:
    zone_runs = z_only[z_only["ZONE"] == zone]
    if len(zone_runs) > 0:
        z_picked.append(zone_runs.head(1))

z_picked = pd.concat(z_picked).sort_values("MINUTES", ascending=False).head(5)
z_picked[["NEXT_MAINT", "PREV_MAINT"]] = z_picked.apply(nearest_maintenance, axis=1)
z_picked["SOURCE"] = "Z만"

# 조용한 구간: 모든 구역이 통계 후보도, IsolationForest 표시도 없는 분
quiet = pd.Series(True, index=df.index)
for zone in GROUPS:
    quiet = quiet & (df[zone + "__STAT_LABEL"] != 1) & (df[zone + "__IF_FLAG"] == 0)

quiet_starts = df[quiet].sample(5, random_state=RANDOM_STATE).sort_index()

quiet_rows = []
for i in quiet_starts.index:
    lot = df.loc[i, "LOT_NO"]
    part = df[(df["LOT_NO"] == lot) & (df.index >= i)].head(10)

    quiet_rows.append({
        "LOT_NO": int(lot),
        "GRADE": part["GRADE"].iloc[0],
        "ZONE": "전체",
        "SENSOR": "",
        "START": part["MEAS_DT"].min(),
        "END": part["MEAS_DT"].max() + pd.Timedelta(seconds=59),
        "MINUTES": len(part),
        "IF_SCORE": np.nan,
        "MAX_ZPAST": np.nan,
        "STAT_ANY": 0,
        "ZSCORE_ANY": 0,
        "DECIDED": "",
        "SOURCE": "조용",
    })

quiet_picked = pd.DataFrame(quiet_rows)
quiet_picked[["NEXT_MAINT", "PREV_MAINT"]] = quiet_picked.apply(nearest_maintenance, axis=1)

print("Z-score만:", len(z_picked), "/ 조용한 구간:", len(quiet_picked))

## 11. 저장

| 컬럼 | 뜻 |
| --- | --- |
| `CANDIDATE_ID` | M001부터. 검토 사이트와 검토 표에서 이 번호로 부름 |
| `SOURCE` | IF, Z만, 조용 (팀에게는 보여 주지 않음) |
| `IF_SCORE` | 낮을수록 평소와 다름 |
| `MAX_ZPAST`, `SENSOR` | 묶음 안에서 가장 큰 지난 값 \|Z\|와 그 센서 |
| `STAT_ANY`, `ZSCORE_ANY` | 통계 후보가 하나라도 있는지, \|Z\| > 3이 하나라도 있는지 |
| `NEXT_MAINT`, `PREV_MAINT` | 가까운 CAL01 정비 기록 |

In [ ]:
output = pd.concat([candidates, z_picked, quiet_picked]).sort_values(["LOT_NO", "START"]).reset_index(drop=True)

ids = []
for i in range(len(output)):
    ids.append("M" + str(i + 1).zfill(3))
output.insert(0, "CANDIDATE_ID", ids)

save_columns = ["CANDIDATE_ID", "SOURCE", "LOT_NO", "GRADE", "ZONE", "SENSOR", "START", "END", "MINUTES",
                "IF_SCORE", "MAX_ZPAST", "STAT_ANY", "ZSCORE_ANY", "NEXT_MAINT", "PREV_MAINT"]
output = output[save_columns]
output.to_csv(OUT_CSV, index=False, encoding=ENC)

print("저장:", OUT_CSV, "/", len(output), "개")
output.groupby("SOURCE")["CANDIDATE_ID"].count()

## 12. 후보 목록

In [ ]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
output